# Preparation for MLE_pipeline

This code give you the parameters you want to start the MLE_pipeline

In [2]:
import sys
sys.path.append('/home/victor-glorieux/Internship_Victor_CBC_ET')
sys.path.append('/home/victor/Internship_Victor_CBC_ET')
sys.path.append('/home/victor-glorieux/Internship_Victor_CBC_ET/code_Adrian/MLE_pipeline/src')
sys.path.append('/home/victor/Internship_Victor_CBC_ET/code_Adrian/MLE_pipeline/src')

from get_data import read_MDC_data
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['font.size'] = 14  # global font size
from pycbc.types import TimeSeries as PycbcTimeSeries
from pycbc.conversions import mchirp_from_mass1_mass2, q_from_mass1_mass2, mass1_from_mchirp_q, mass2_from_mchirp_q
from fonctions import extraction_temps, extract_mchirp_tc_spectro

In [ ]:
study_index = 0
study_type = 3

init, final, t0_list, tc_list, interval, params_list = extraction_temps(indexes = [study_index], type = study_type,
                                                         source="IJCLab_server", print_ = False)

print(r't0 :',t0_list[0])
print(r'tc :',tc_list[0])

In [ ]:
#==========
pourcentage = 70 #Pourcentage de la durée du signal que nous supprimons depuis t0.
#==========

Delta = ((tc_list[0] - t0_list[0])/100) * pourcentage
t_start = t0_list[0] + Delta
t_stop = tc_list[0] + 0.2

ifos=['E1', 'E2', 'E3']
data = read_MDC_data(t_start, t_stop)
signal = {}
for ifo in ifos:
    #data[ifo] = data[ifo].resample(4096)
    data[ifo] = data[ifo].resample(2048)
    #data[ifo] = data[ifo].resample(1024)
    val = data[ifo].value
    delta_t = data[ifo].dt.value
    pycbc_ts = PycbcTimeSeries(val, delta_t=delta_t)
    signal[ifo] = pycbc_ts.to_frequencyseries()

print('Durations = ',t_stop - t_start)

In [ ]:
ET_params = pd.read_csv("/home/victor-glorieux/Internship_Victor_CBC_ET/code_Adrian/MLE_pipeline/data/loudest_BBH/list_mdc1_v2.txt",sep = ' ',engine='python', index_col = False)
ET_params = ET_params.sort_values('snr',ascending=False) #Sélectionne les events avec le meilleur SNR pour les indices les plus faibles.
ET_params = ET_params[ET_params['type'] == study_type] #Sélectionne un type particulier d'événements.
params_list = ET_params.iloc[study_index]
print(params_list)
mchirp_signal = mchirp_from_mass1_mass2(params_list.mz1,params_list.mz2)
q_signal = q_from_mass1_mass2(params_list.mz1,params_list.mz2)

print(r'Chirp mass :',mchirp_signal)
print(r'Corrected chirp mass :',mchirp_signal * (1+params_list.z))
print(r'tc :',params_list.tc)

In [ ]:
colorbar_limits = {'inf' : 2, 'sup' : 60}
dict = extract_mchirp_tc_spectro(data,'E1',colorbar_limits=colorbar_limits,q_lim = 15,frange=(4, 150),qrange=(10, 50),
                                fres=0.1, tres=10,show_fit=True)
print(r'Fitted chirp mass : {}, error : {}.'.format(dict["mchirp"],dict["u_mchirp"]))
print(r'Fitted tc : {}, error : {}.'.format(dict["tc"],dict["u_tc"]))